## ДЗ 2.3. Связанные колебания и жёсткость

### Постановка задачи

Рассматриваются две одинаковые материальные точки массы $m$, каждая из которых соединена пружиной жесткости $k_0$ с неподвижной опорой. Между массами размещены дополнительная пружина жесткости $k_c$ и демпфер с коэффициентом вязкого трения $c$.

Смещения масс от положения статического равновесия описываются системой дифференциальных уравнений:

$$\begin{aligned}
m \ddot{x}_1 &= -k_0 x_1 - k_c(x_1 - x_2) - c(\dot{x}_1 - \dot{x}_2), \\
m \ddot{x}_2 &= -k_0 x_2 - k_c(x_2 - x_1) - c(\dot{x}_2 - \dot{x}_1).
\end{aligned}$$

Система переписывается в форме Коши первого порядка для вектора состояния $\mathbf{y} = [x_1, v_1, x_2, v_2]^T$:

$$\frac{d\mathbf{y}}{dt} = \mathbf{A}\mathbf{y}, \qquad
\mathbf{A} = \begin{pmatrix}
0 & 1 & 0 & 0 \\
-\frac{k_0 + k_c}{m} & -\frac{c}{m} & \frac{k_c}{m} & \frac{c}{m} \\
0 & 0 & 0 & 1 \\
\frac{k_c}{m} & \frac{c}{m} & -\frac{k_0 + k_c}{m} & -\frac{c}{m}
\end{pmatrix}.$$

### Анализ нормальных мод системы

Перейдем к нормальным координатам — синфазной $u = x_1 + x_2$ и противофазной (относительной) $v = x_1 - x_2$:

1. **Синфазная мода $u = x_1 + x_2$:**  
   Складывая уравнения движения, получаем:
   $$m \ddot{u} + k_0 u = 0 \implies \ddot{u} + \omega_0^2 u = 0, \quad \omega_0 = \sqrt{\frac{k_0}{m}}.$$
   *Важнейшее физическое свойство:* соединительная пружина $k_c$ и демпфер $c$ действуют только на относительное перемещение $x_1 - x_2$. При синхронном движении масс пружина связи не деформируется, а демпфер неподвижен. Поэтому синфазная мода колеблется с неизменной частотой $\omega_0$ и **не затухает**.

2. **Противофазная (относительная) мода $v = x_1 - x_2$:**  
   Вычитая уравнения движения, получаем:
   $$m \ddot{v} + 2c \dot{v} + (k_0 + 2k_c) v = 0 \implies \ddot{v} + 2\gamma \dot{v} + \omega_{\mathrm{rel}}^2 v = 0,$$
   где $\gamma = c/m$ — коэффициент затухания, а $\omega_{\mathrm{rel}} = \sqrt{(k_0 + 2k_c)/m}$.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from matplotlib.patches import Rectangle
from IPython.display import HTML

# Оформление графиков
plt.rcParams.update({
    "font.size": 11,
    "axes.grid": True,
    "grid.alpha": 0.3,
    "figure.titlesize": 13,
    "axes.titlesize": 12,
    "lines.linewidth": 1.5
})


## 1. Численные методы интегрирования

В работе сопоставляются три схемы:
1. **RK4 (классический метод Рунге-Кутты 4-го порядка):**  
   Явный четырехстадийный метод 4-го порядка. Область абсолютной устойчивости ограничена по мнимой оси отрезком $[-2.83i, 2.83i]$. Для незатухающих колебаний отсюда следует $h\omega\le 2\sqrt{2}$. Если собственные значения имеют отрицательную вещественную часть, границу нужно искать по условию $|R_{\mathrm{RK4}}(h\lambda_j)|\le1$ для всех мод, где $R_{\mathrm{RK4}}(z)=1+z+z^2/2+z^3/6+z^4/24$.
2. **Скоростной метод Верле (Velocity Verlet):**  
   Симплектический метод 2-го порядка для систем ньютоновского типа $\ddot{\mathbf{x}} = \mathbf{a}(\mathbf{x})$. Идеально сохраняет фазовый объем и квазиинтеграл энергии при отсутствии диссипации ($c = 0$).
3. **Неявный метод трапеций (семейство Адамса–Моултона 2-го порядка / Crank-Nicolson):**  
   $$\mathbf{y}_{n+1} = \mathbf{y}_n + \frac{h}{2}\bigl(\mathbf{A}\mathbf{y}_n + \mathbf{A}\mathbf{y}_{n+1}\bigr) \implies \mathbf{y}_{n+1} = \left(\mathbf{I} - \frac{h}{2}\mathbf{A}\right)^{-1} \left(\mathbf{I} + \frac{h}{2}\mathbf{A}\right) \mathbf{y}_n.$$
   Метод трапеций является **A-устойчивым**: его функция устойчивости $R(z) = \frac{1 + z/2}{1 - z/2}$ удовлетворяет условию $|R(z)| \le 1$ во всей левой полуплоскости $\mathrm{Re}(z) \le 0$. Численная схема устойчива при **любом** шаге интегрирования $h > 0$, но это не гарантирует правильные частоты и фазу колебаний.


In [ ]:
def get_system_matrix(m, k0, kc, c):
    """Формирование матрицы A линейной системы y' = A y."""
    return np.array([
        [0.0, 1.0, 0.0, 0.0],
        [-(k0 + kc)/m, -c/m, kc/m, c/m],
        [0.0, 0.0, 0.0, 1.0],
        [kc/m, c/m, -(k0 + kc)/m, -c/m]
    ], dtype=np.float64)

def step_rk4(y, h, A):
    """Шаг классического явного метода Рунге-Кутты 4-го порядка."""
    k1 = A @ y
    k2 = A @ (y + 0.5 * h * k1)
    k3 = A @ (y + 0.5 * h * k2)
    k4 = A @ (y + h * k3)
    return y + (h / 6.0) * (k1 + 2.0 * k2 + 2.0 * k3 + k4)

def step_trapezoidal(y, h, A):
    """Шаг неявного A-устойчивого метода трапеций (Адамс-Моултон 2 порядка)."""
    I = np.eye(len(y))
    M_left = I - 0.5 * h * A
    M_right = I + 0.5 * h * A
    return np.linalg.solve(M_left, M_right @ y)

def step_verlet_conservative(x1, v1, x2, v2, h, m, k0, kc):
    """Шаг скоростного метода Верле для консервативной системы (c = 0)."""
    a1 = (-k0 * x1 - kc * (x1 - x2)) / m
    a2 = (-k0 * x2 - kc * (x2 - x1)) / m
    x1_new = x1 + h * v1 + 0.5 * h**2 * a1
    x2_new = x2 + h * v2 + 0.5 * h**2 * a2
    a1_new = (-k0 * x1_new - kc * (x1_new - x2_new)) / m
    a2_new = (-k0 * x2_new - kc * (x2_new - x1_new)) / m
    v1_new = v1 + 0.5 * h * (a1 + a1_new)
    v2_new = v2 + 0.5 * h * (a2 + a2_new)
    return x1_new, v1_new, x2_new, v2_new

def simulate_ode(step_func, y0, h, t_end, A=None):
    """Универсальный цикл интегрирования с постоянным шагом h."""
    n_steps = int(round(t_end / h))
    t_arr = np.linspace(0.0, n_steps * h, n_steps + 1)
    y_arr = np.empty((n_steps + 1, len(y0)), dtype=np.float64)
    y_arr[0] = y0
    for i in range(n_steps):
        if A is not None:
            y_arr[i + 1] = step_func(y_arr[i], h, A)
        else:
            y_arr[i + 1] = step_func(y_arr[i], h)
    return t_arr, y_arr


---
## 2. Биения при слабой связи ($k_c \ll k_0, c = 0$)

### Аналитическое решение

При слабой связи $k_c \ll k_0$ и отсутствии демпфирования ($c = 0$) собственные частоты системы равны:
$$\omega_1 = \sqrt{\frac{k_0}{m}}, \qquad \omega_2 = \sqrt{\frac{k_0 + 2k_c}{m}}.$$

При начальных условиях $x_1(0) = A_0, x_2(0) = 0, v_1(0) = v_2(0) = 0$ точное решение выражается как суперпозиция двух нормальных колебаний:
$$\begin{aligned}
x_1(t) &= \frac{A_0}{2}\bigl(\cos\omega_1 t + \cos\omega_2 t\bigr) = A_0 \cos(\omega_{\mathrm{б}} t) \cos(\bar{\omega} t), \\
x_2(t) &= \frac{A_0}{2}\bigl(\cos\omega_1 t - \cos\omega_2 t\bigr) = A_0 \sin(\omega_{\mathrm{б}} t) \sin(\bar{\omega} t),
\end{aligned}$$
где несущая частота $\bar{\omega} = (\omega_1 + \omega_2)/2$, а частота биений $\omega_{\mathrm{б}} = (\omega_2 - \omega_1)/2$.  
Период биений огибающей: $T_{\mathrm{б}} = 2\pi / \omega_{\mathrm{б}}$, а период полной перекачки энергии от одной массы к другой равен $T_{\mathrm{trans}} = T_{\mathrm{б}} / 2$.


In [ ]:
# Физические параметры для зеленого уровня
m_gr = 1.0
k0_gr = 10.0
kc_gr = 0.1       # слабая связь: kc / k0 = 0.01 << 1
c_gr = 0.0        # демпфирование отсутствует
A0 = 1.0

# Собственные частоты и периоды
omega1_gr = np.sqrt(k0_gr / m_gr)
omega2_gr = np.sqrt((k0_gr + 2.0 * kc_gr) / m_gr)
omega_beat_gr = (omega2_gr - omega1_gr) / 2.0
T_beat_gr = 2.0 * np.pi / omega_beat_gr
T_transfer_gr = T_beat_gr / 2.0

print(f"Собственная частота 1 (синфазная):      omega_1 = {omega1_gr:.4f} рад/с")
print(f"Собственная частота 2 (противофазная):  omega_2 = {omega2_gr:.4f} рад/с")
print(f"Частота биений огибающей:               omega_б = {omega_beat_gr:.4f} рад/с")
print(f"Полный период биений:                   T_б     = {T_beat_gr:.2f} с")
print(f"Период перекачки энергии (T_б / 2):     T_trans = {T_transfer_gr:.2f} с")

# Начальное состояние [x1, v1, x2, v2]
y0_gr = np.array([A0, 0.0, 0.0, 0.0], dtype=np.float64)
A_matrix_gr = get_system_matrix(m_gr, k0_gr, kc_gr, c_gr)

def exact_solution_green(t):
    """Точное аналитическое решение для смещений и скоростей."""
    x1 = 0.5 * A0 * (np.cos(omega1_gr * t) + np.cos(omega2_gr * t))
    x2 = 0.5 * A0 * (np.cos(omega1_gr * t) - np.cos(omega2_gr * t))
    v1 = -0.5 * A0 * (omega1_gr * np.sin(omega1_gr * t) + omega2_gr * np.sin(omega2_gr * t))
    v2 = -0.5 * A0 * (omega1_gr * np.sin(omega1_gr * t) - omega2_gr * np.sin(omega2_gr * t))
    return x1, v1, x2, v2

# Расчет на интервале трех полных периодов биений для набора шагов
t_end_gr = 3.0 * T_beat_gr
dt_steps_gr = [0.05, 0.02, 0.01, 0.005]
solutions_gr = {}

for dt in dt_steps_gr:
    # 1. RK4
    t_rk, y_rk = simulate_ode(step_rk4, y0_gr, dt, t_end_gr, A_matrix_gr)
    
    # 2. Скоростной Верле
    n_steps = int(round(t_end_gr / dt))
    t_ver = np.linspace(0.0, n_steps * dt, n_steps + 1)
    y_ver = np.empty((n_steps + 1, 4), dtype=np.float64)
    y_ver[0] = y0_gr
    for i in range(n_steps):
        x1_n, v1_n, x2_n, v2_n = y_ver[i]
        y_ver[i + 1] = step_verlet_conservative(x1_n, v1_n, x2_n, v2_n, dt, m_gr, k0_gr, kc_gr)
        
    solutions_gr[("RK4", dt)] = (t_rk, y_rk)
    solutions_gr[("Verlet", dt)] = (t_ver, y_ver)


### Артефакт 1: Графики смещений обеих масс (биения)

Ниже представлены траектории $x_1(t)$ и $x_2(t)$ обеих масс. На графиках отчетливо видна периодическая перекачка энергии: в моменты времени, кратные $T_{\mathrm{trans}} \approx 200$ с, амплитуда колебаний первой массы обращается в ноль, а вторая масса колеблется с максимальной амплитудой $A_0$, после чего процесс обращается.


In [ ]:
fig, axes = plt.subplots(len(dt_steps_gr), 2, figsize=(14, 3.0 * len(dt_steps_gr)),
                         sharex=True, constrained_layout=True)
fig.suptitle("Смещения двух масс при слабой связи (явление биений)", fontsize=13, y=1.02)

for row, dt in enumerate(dt_steps_gr):
    t_cur, y_rk = solutions_gr[("RK4", dt)]
    _, y_ver = solutions_gr[("Verlet", dt)]
    x1_ex, _, x2_ex, _ = exact_solution_green(t_cur)
    
    # Масса 1
    ax1 = axes[row, 0]
    ax1.plot(t_cur, x1_ex, 'k-', lw=1.2, alpha=0.35, label='Аналитическое')
    ax1.plot(t_cur, y_rk[:, 0], '-', color='tab:blue', lw=0.9, label='RK4')
    ax1.plot(t_cur, y_ver[:, 0], '--', color='tab:orange', lw=0.9, label='Верле')
    ax1.set_ylabel(r"$x_1(t)$, м")
    ax1.set_title(f"Масса 1: $x_1(t)$, шаг $h = {dt}$ с")
    ax1.legend(loc='upper right', fontsize=8)
    
    # Масса 2
    ax2 = axes[row, 1]
    ax2.plot(t_cur, x2_ex, 'k-', lw=1.2, alpha=0.35, label='Аналитическое')
    ax2.plot(t_cur, y_rk[:, 2], '-', color='tab:blue', lw=0.9, label='RK4')
    ax2.plot(t_cur, y_ver[:, 2], '--', color='tab:orange', lw=0.9, label='Верле')
    ax2.set_ylabel(r"$x_2(t)$, м")
    ax2.set_title(f"Масса 2: $x_2(t)$, шаг $h = {dt}$ с")
    ax2.legend(loc='upper right', fontsize=8)

axes[-1, 0].set_xlabel("Время $t$, с")
axes[-1, 1].set_xlabel("Время $t$, с")
plt.show()


### Артефакт 2: Графики кинетической энергии каждой массы и полной механической энергии

Полная механическая энергия консервативной системы состоит из суммы кинетических энергий обеих масс и потенциальной энергии всех трех пружин:
$$T_1 = \frac{1}{2}m v_1^2, \quad T_2 = \frac{1}{2}m v_2^2, \quad U = \frac{1}{2}k_0(x_1^2 + x_2^2) + \frac{1}{2}k_c(x_1 - x_2)^2, \quad E_{\mathrm{полн}} = T_1 + T_2 + U.$$


In [ ]:
def compute_energies(y_states, m, k0, kc):
    """Расчет кинетических энергий T1, T2, потенциальной U и полной энергии E."""
    x1, v1, x2, v2 = y_states[:, 0], y_states[:, 1], y_states[:, 2], y_states[:, 3]
    T1 = 0.5 * m * v1**2
    T2 = 0.5 * m * v2**2
    U = 0.5 * k0 * (x1**2 + x2**2) + 0.5 * kc * (x1 - x2)**2
    E_tot = T1 + T2 + U
    return T1, T2, U, E_tot

# Начальная энергия
E0_gr = 0.5 * k0_gr * A0**2 + 0.5 * kc_gr * A0**2

fig, axes = plt.subplots(len(dt_steps_gr), 2, figsize=(14, 3.0 * len(dt_steps_gr)),
                         sharex=True, constrained_layout=True)
fig.suptitle("Энергетический баланс системы: кинетические энергии и полная энергия", fontsize=13, y=1.02)

for row, dt in enumerate(dt_steps_gr):
    for col, (method_name, col_color) in enumerate([("RK4", "tab:blue"), ("Verlet", "tab:orange")]):
        ax = axes[row, col]
        t_cur, y_sol = solutions_gr[(method_name, dt)]
        T1, T2, U, E_tot = compute_energies(y_sol, m_gr, k0_gr, kc_gr)
        
        ax.plot(t_cur, T1, lw=0.8, color='tab:blue', alpha=0.8, label=r'$T_1$')
        ax.plot(t_cur, T2, lw=0.8, color='tab:red', alpha=0.8, label=r'$T_2$')
        ax.plot(t_cur, E_tot, 'k-', lw=1.2, label=r'$E_{\mathrm{полн}}$')
        
        ax.set_title(f"{method_name}, шаг $h = {dt}$ с")
        ax.set_ylabel("Энергия, Дж")
        ax.legend(loc='upper right', fontsize=8)

axes[-1, 0].set_xlabel("Время $t$, с")
axes[-1, 1].set_xlabel("Время $t$, с")
plt.show()


### Сохранение энергии и сходимость по шагу $h$

Сопоставим относительное отклонение полной энергии $\Delta E / E_0 = (E(t) - E_0) / E_0$:
- Для **RK4** наблюдается монотонный дрейф погрешности, убывающий пропорционально $O(h^4)$.
- Для **метода Верле** (симплектического) дрейфа энергии нет: погрешность колеблется около нуля в узком диапазоне $O(h^2)$, обеспечивая превосходное долгосрочное сохранение первого интеграла.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.2), constrained_layout=True)
fig.suptitle("Относительное отклонение полной энергии $(E(t) - E_0)/E_0$", fontsize=12)

colors_dt = ['tab:red', 'tab:green', 'tab:blue', 'tab:purple']

for col, method in enumerate(["RK4", "Verlet"]):
    ax = axes[col]
    for dt, color in zip(dt_steps_gr, colors_dt):
        t_cur, y_sol = solutions_gr[(method, dt)]
        _, _, _, E_tot = compute_energies(y_sol, m_gr, k0_gr, kc_gr)
        rel_err = (E_tot - E0_gr) / E0_gr
        ax.plot(t_cur, rel_err, lw=0.9, color=color, label=f"h = {dt} с")
    ax.set_title(f"Метод: {method}")
    ax.set_xlabel("Время $t$, с")
    ax.set_ylabel(r"$(E(t) - E_0)/E_0$")
    ax.legend(fontsize=8)
    ax.ticklabel_format(axis='y', style='sci', scilimits=(-3, 3))

plt.show()

# Сводная таблица сходимости
print(f"{'Метод':<8} {'h, с':<8} {'max|ΔE|/E0':<16} {'max|x1 - x1_exact|, м':<24}")
print("-" * 58)
for method in ["RK4", "Verlet"]:
    for dt in dt_steps_gr:
        t_cur, y_sol = solutions_gr[(method, dt)]
        _, _, _, E_tot = compute_energies(y_sol, m_gr, k0_gr, kc_gr)
        max_de = np.max(np.abs(E_tot - E0_gr)) / E0_gr
        x1_ex, _, _, _ = exact_solution_green(t_cur)
        max_dx = np.max(np.abs(y_sol[:, 0] - x1_ex))
        print(f"{method:<8} {dt:<8.3f} {max_de:<16.3e} {max_dx:<24.3e}")


### Спектральный анализ (FFT) и сравнение собственных частот

Выполним быстрое преобразование Фурье для траектории $x_1(t)$ на длительном интервале времени ($10 T_{\mathrm{б}}$).
В спектре отчетливо выделяются два резких пика на частотах нормальных мод $\omega_1$ и $\omega_2$.


In [ ]:
t_long_gr = 10.0 * T_beat_gr
dt_fft = 0.005
t_fft_arr, y_fft_arr = simulate_ode(step_rk4, y0_gr, dt_fft, t_long_gr, A_matrix_gr)
x1_fft_signal = y_fft_arr[:, 0]

# БПФ с окном Ханна
N_pts = len(x1_fft_signal)
window = np.hanning(N_pts)
spectrum = np.abs(np.fft.rfft(x1_fft_signal * window))
freqs_hz = np.fft.rfftfreq(N_pts, d=dt_fft)
omegas_rad = 2.0 * np.pi * freqs_hz

# Поиск двух доминирующих пиков в окрестности omega1
mask = (omegas_rad > 2.0) & (omegas_rad < 4.5)
sub_omegas = omegas_rad[mask]
sub_spec = spectrum[mask]
peak_indices = np.argsort(sub_spec)[-2:]
peak_omegas = np.sort(sub_omegas[peak_indices])

num_w1, num_w2 = peak_omegas[0], peak_omegas[1]
num_w_beat = (num_w2 - num_w1) / 2.0
num_T_beat = 2.0 * np.pi / num_w_beat

print("=" * 65)
print("СОПОСТАВЛЕНИЕ ЧАСТОТ (АНАЛИТИЧЕСКИЕ vs ЧИСЛЕННЫЕ ИЗ FFT):")
print(f"  Синфазная частота omega_1:      аналит = {omega1_gr:.5f},  числ = {num_w1:.5f} рад/с")
print(f"  Противофазная частота omega_2:  аналит = {omega2_gr:.5f},  числ = {num_w2:.5f} рад/с")
print(f"  Частота биений omega_б:         аналит = {omega_beat_gr:.5f},  числ = {num_w_beat:.5f} рад/с")
print(f"  Период биений T_б:              аналит = {T_beat_gr:.2f} с,  числ = {num_T_beat:.2f} с")
print("=" * 65)

fig, ax = plt.subplots(figsize=(10, 4.2), constrained_layout=True)
ax.plot(omegas_rad, spectrum / np.max(spectrum), 'tab:blue', lw=1.2, label='Спектр $x_1(t)$')
ax.axvline(omega1_gr, color='tab:red', ls='--', lw=1.2, label=rf'$\omega_1 = {omega1_gr:.3f}$ рад/с')
ax.axvline(omega2_gr, color='tab:green', ls='--', lw=1.2, label=rf'$\omega_2 = {omega2_gr:.3f}$ рад/с')
ax.set_xlim(omega1_gr - 0.25, omega2_gr + 0.25)
ax.set_xlabel(r"Частота $\omega$, рад/с")
ax.set_ylabel("Нормированная амплитуда")
ax.set_title(r"Спектр Фурье: расщепление собственных частот системы на $\Delta\omega = 2\omega_{\mathrm{б}}$")
ax.legend(loc='upper right')
plt.show()


---
## 3. Сильная связь, демпфирование и жёсткость системы

### Физическая постановка задачи
Увеличим жесткость соединительной пружины $k_c \gg k_0$ и введем демпфер между массами ($c > 0$).
В такой системе возникают два принципиально разных масштаба времени:
1. **Быстрый переходный процесс:** интенсивные высокочастотные колебания масс навстречу друг другу с собственной частотой $\omega_{\mathrm{fast}} = \sqrt{(k_0 + 2k_c)/m}$ (при демпфировании частота колебаний равна $\Omega=\sqrt{\omega_{\mathrm{fast}}^2-\gamma^2}$), быстро гасимые демпфером с декрементом $\gamma = c/m$.
2. **Медленное движение:** после практического затухания относительного движения $x_1(t) - x_2(t) \to 0$ массы совершают совместные незатухающие гармонические колебания на медленной частоте $\omega_{\mathrm{slow}} = \sqrt{k_0/m}$.

### Выбор параметров
Выберем параметры системы:
$$m = 1.0, \quad k_0 = 1.0, \quad k_c = 200.0, \quad c = 12.0.$$
- Медленная частота: $\omega_{\mathrm{slow}} = \sqrt{1.0/1.0} = 1.0$ рад/с, медленный период $T_{\mathrm{slow}} = 2\pi \approx 6.28$ с.
- Быстрая частота: $\omega_{\mathrm{fast}} = \sqrt{(1.0 + 2 \cdot 200.0)/1.0} = \sqrt{401} \approx 20.025$ рад/с, период $T_{\mathrm{fast}} \approx 0.314$ с.
- С учетом демпфера: $\Omega=\sqrt{401-144}=\sqrt{257}\approx16.031$ рад/с, период затухающих колебаний $2\pi/\Omega\approx0.392$ с.
- Коэффициент жесткости системы: $S = \omega_{\mathrm{fast}} / \omega_{\mathrm{slow}} \approx 20$.
- Время затухания быстрой моды: $\tau = m / c = 1 / 12 \approx 0.083$ с. За время $t = 1.0$ с огибающая относительного движения уменьшается в $e^{12} \approx 1.6 \cdot 10^5$ раз.


In [ ]:
m_y = 1.0
k0_y = 1.0
kc_y = 200.0
c_y = 12.0

A_matrix_y = get_system_matrix(m_y, k0_y, kc_y, c_y)
eigvals_y = np.linalg.eigvals(A_matrix_y)

omega_slow_y = np.sqrt(k0_y / m_y)
omega_fast_y = np.sqrt((k0_y + 2.0 * kc_y) / m_y)
gamma_y = c_y / m_y
Omega_y = np.sqrt(omega_fast_y**2 - gamma_y**2)

def R_rk4(z):
    return 1 + z + z**2 / 2 + z**3 / 6 + z**4 / 24

# Ищем первую границу |R(h*lambda)| = 1 вдоль луча каждой моды.
limits_rk4 = []
for lam in eigvals_y:
    lo, hi = 0.0, 1.0 / abs(lam)
    while abs(R_rk4(hi * lam)) <= 1.0:
        hi *= 2.0
    for _ in range(60):
        mid = 0.5 * (lo + hi)
        if abs(R_rk4(mid * lam)) <= 1.0:
            lo = mid
        else:
            hi = mid
    limits_rk4.append(0.5 * (lo + hi))
h_crit_rk4 = min(limits_rk4)

def exact_solution_y(t):
    """Решение для x1(0)=1, x2(0)=v1(0)=v2(0)=0."""
    t = np.asarray(t)
    u = np.cos(omega_slow_y * t)
    du = -omega_slow_y * np.sin(omega_slow_y * t)
    v = np.exp(-gamma_y * t) * (np.cos(Omega_y * t)
                                + gamma_y / Omega_y * np.sin(Omega_y * t))
    dv = -(omega_fast_y**2 / Omega_y) * np.exp(-gamma_y * t) * np.sin(Omega_y * t)
    return np.stack(((u + v)/2, (du + dv)/2, (u - v)/2, (du - dv)/2), axis=-1)

print("СПЕКТРАЛЬНЫЙ АНАЛИЗ ЖЕСТКОЙ СИСТЕМЫ:")
print(f"  Медленная мода:  omega_slow = {omega_slow_y:.4f} рад/с,  T_slow = {2*np.pi/omega_slow_y:.2f} с")
print(f"  Быстрая мода:    omega_fast = {omega_fast_y:.4f} рад/с,  T_fast = {2*np.pi/omega_fast_y:.4f} с")
print(f"  Частота с демпфером: Omega = {Omega_y:.4f} рад/с, период = {2*np.pi/Omega_y:.4f} с")
print(f"  Отношение частот (параметр жесткости): S = {omega_fast_y / omega_slow_y:.1f}")
print("\nСобственные значения матрицы оператора A:")
for ev in eigvals_y:
    print(f"  lambda = {ev.real:+.4f} {ev.imag:+.4f}j")
print(f"\nКритический теоретический шаг устойчивости для RK4: h_crit ≈ {h_crit_rk4:.4f} с")

print("\nПроверка коэффициента усиления быстрой моды:")
lam_fast = complex(-gamma_y, Omega_y)
print(f"{'h, с':>8} {'|R(h*lambda_fast)|':>22}  Результат")
for h in [0.12, 0.13, 0.132, 0.14, 0.20]:
    amplification = abs(R_rk4(h * lam_fast))
    print(f"{h:8.3f} {amplification:22.6f}  "
          + ("затухание" if amplification < 1 else "рост"))


### Артефакт 3: Графики решений явного (RK4) и неявного (трапеций) методов

Исследуем шаги $h=0.01,\ 0.05,\ 0.12,\ 0.13,\ 0.14,\ 0.20$ с.
Для спектра $\lambda=\pm i,\ -12\pm i\sqrt{257}$ граница RK4 равна $h_*\approx0.131258$ с.
Поэтому шаг можно увеличить с $0.12$ до $0.13$ с, сохранив устойчивость; при $0.14$ с быстрая мода уже растет.
Прежняя оценка $2.8/\max|\operatorname{Im}\lambda|\approx0.175$ с здесь неприменима: она учитывала только мнимую часть спектра.

Слева показано относительное смещение на начальном участке $0\le t\le2$ с, справа — движение первой массы на всем интервале до $15$ с. Точная кривая позволяет увидеть, что устойчивое решение тоже может плохо передавать переходный процесс.


In [ ]:
y0_y = np.array([1.0, 0.0, 0.0, 0.0], dtype=np.float64)
test_steps_y = [0.01, 0.05, 0.12, 0.13, 0.14, 0.20]
t_end_y = 15.0

fig, axes = plt.subplots(len(test_steps_y), 2, figsize=(14, 3.2 * len(test_steps_y)),
                         constrained_layout=True)
fig.suptitle("Сравнение явного RK4 и неявного метода трапеций AM2 при различных шагах", fontsize=13, y=1.02)

for row, h_val in enumerate(test_steps_y):
    # 1. Интегрирование методом трапеций
    t_trap, y_trap = simulate_ode(step_trapezoidal, y0_y, h_val, t_end_y, A_matrix_y)
    
    # 2. Интегрирование явным методом RK4 с контролем переполнения
    n_pts_rk = int(round(t_end_y / h_val))
    t_rk = np.linspace(0.0, n_pts_rk * h_val, n_pts_rk + 1)
    y_rk = np.empty((n_pts_rk + 1, 4), dtype=np.float64)
    y_rk[0] = y0_y
    rk_exploded = False
    for step_i in range(n_pts_rk):
        y_next = step_rk4(y_rk[step_i], h_val, A_matrix_y)
        if np.any(np.isnan(y_next)) or np.any(np.abs(y_next) > 1e4):
            y_rk[step_i + 1:] = np.nan
            rk_exploded = True
            break
        y_rk[step_i + 1] = y_next

    # Левая колонка: Начальный переходный процесс (t in [0, 2] c)
    ax_trans = axes[row, 0]
    t_exact = np.linspace(0.0, t_end_y, 3001)
    y_exact = exact_solution_y(t_exact)
    initial = t_exact <= 2.0
    ax_trans.plot(t_exact[initial], y_exact[initial, 0] - y_exact[initial, 2],
                  'k:', lw=1.3, label='Точное решение')
    mask_trans = (t_trap <= 2.0)
    ax_trans.plot(t_trap[mask_trans], y_trap[mask_trans, 0] - y_trap[mask_trans, 2],
                  'o-', color='tab:green', ms=3, lw=1.2, label=r'$x_1 - x_2$ (Трапеции)')
    if not rk_exploded or np.nanmax(t_rk[~np.isnan(y_rk[:, 0])]) >= 2.0:
        mask_trans_rk = (t_rk <= 2.0) & (~np.isnan(y_rk[:, 0]))
        ax_trans.plot(t_rk[mask_trans_rk], y_rk[mask_trans_rk, 0] - y_rk[mask_trans_rk, 2],
                      'o--', color='tab:blue', ms=3, lw=1.2, label=r'$x_1 - x_2$ (RK4)')
    ax_trans.set_ylim(-0.5, 1.15)
    ax_trans.set_title(f"Переходный процесс ($x_1 - x_2$), h = {h_val} с")
    ax_trans.set_ylabel("Относит. смещение, м")
    ax_trans.set_xlabel("Время $t$, с")
    ax_trans.legend(loc='upper right', fontsize=8)
    
    # Правая колонка: Медленное движение (t in [0, 15] c)
    ax_slow = axes[row, 1]
    ax_slow.plot(t_exact, y_exact[:, 0], 'k:', lw=1.3, label='Точное решение')
    ax_slow.plot(t_trap, y_trap[:, 0], 'tab:green', lw=1.5, label=r'$x_1$ (Трапеции AM2)')
    if rk_exploded:
        valid_rk = ~np.isnan(y_rk[:, 0])
        ax_slow.plot(t_rk[valid_rk], y_rk[valid_rk, 0], 'r--', lw=1.2, label=r'$x_1$ (RK4 — ВЗРЫВ!)')
        ax_slow.set_ylim(-1.5, 1.5)
    else:
        ax_slow.plot(t_rk, y_rk[:, 0], 'tab:blue', ls='--', lw=1.2, label=r'$x_1$ (RK4)')
    ax_slow.set_title(f"Полное движение $x_1(t)$, h = {h_val} с" + (" [RK4 неустойчив]" if rk_exploded else ""))
    ax_slow.set_ylabel("Смещение $x_1$, м")
    ax_slow.set_xlabel("Время $t$, с")
    ax_slow.legend(loc='upper right', fontsize=8)

plt.show()


### Численный эксперимент: увеличение шага после затухания быстрого процесса

До $t=1.5$ с используем RK4 с шагом $0.005$ с. К этому моменту быстрая мода мала, но еще не равна нулю: ее огибающая имеет порядок $e^{-18}\approx1.5\cdot10^{-8}$.
Далее сравним RK4 при $h=0.12,\ 0.13,\ 0.20$ с и метод трапеций при $h=0.20$ с. Первые два шага лежат внутри области устойчивости RK4, третий — снаружи.
На последнем шаге при необходимости уменьшаем $h$, чтобы все решения закончились ровно при $t=15$ с.


In [ ]:
t_switch = 1.5
h_initial = 0.005
t_final = 15.0

t_stage1, y_stage1 = simulate_ode(step_rk4, y0_y, h_initial, t_switch, A_matrix_y)
y_switch = y_stage1[-1].copy()
print(f"При t = {t_switch} с: x1 - x2 = {y_switch[0] - y_switch[2]:.3e} м, "
      f"v1 - v2 = {y_switch[1] - y_switch[3]:.3e} м/с")

def continue_solution(step_func, h):
    times, states = [t_switch], [y_switch.copy()]
    while times[-1] < t_final - 1e-12:
        dt = min(h, t_final - times[-1])
        y_next = step_func(states[-1], dt, A_matrix_y)
        if not np.all(np.isfinite(y_next)) or np.max(np.abs(y_next)) > 1e4:
            print(f"RK4, h = {h}: рост выше 1e4 при t = {times[-1] + dt:.3f} с")
            break
        times.append(times[-1] + dt)
        states.append(y_next)
    return np.array(times), np.array(states)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4.5), constrained_layout=True)
t_ref = np.linspace(0, t_final, 3001)
ax1.plot(t_ref, exact_solution_y(t_ref)[:, 0], 'k:', label='Точное решение')
ax1.plot(t_stage1, y_stage1[:, 0], color='gray', label='До увеличения шага')

for method, step_func, h, color in [
    ('RK4', step_rk4, 0.12, 'tab:blue'),
    ('RK4', step_rk4, 0.13, 'tab:orange'),
    ('RK4', step_rk4, 0.20, 'tab:red'),
    ('Трапеции', step_trapezoidal, 0.20, 'tab:green')
]:
    t2, y2 = continue_solution(step_func, h)
    label = f'{method}, h = {h}'
    ax1.plot(t2, y2[:, 0], '--', color=color, lw=1.2, label=label)
    ax2.semilogy(t2, np.maximum(np.abs(y2[:, 0] - y2[:, 2]), 1e-16),
                 color=color, label=label)
    u2 = y2[:, 0] + y2[:, 2]
    if abs(t2[-1] - t_final) < 1e-10:
        print(f'{label}: max|u - cos(t)| = {np.max(np.abs(u2 - np.cos(t2))):.3e}')

ax1.axvline(t_switch, color='black', ls=':')
ax1.set_ylim(-1.5, 1.5)
ax1.set(xlabel='Время t, с', ylabel='Смещение x1, м', title='Увеличение шага после t = 1.5 с')
ax2.set(xlabel='Время t, с', ylabel='|x1 - x2|, м', title='Остаточная быстрая мода: затухание или рост')
for ax in (ax1, ax2):
    ax.legend(fontsize=8)
plt.show()


### Фаза медленных колебаний при крупном шаге трапеций

Сравним $u=x_1+x_2$ с точным $u(t)=\cos t$ после увеличения шага. Здесь быстрая мода исключается самим сложением уравнений, поэтому видна именно ошибка медленных колебаний. В таблице указаны максимальная ошибка $u$ в узлах и отставание фазы за $13.5$ с. Последний укороченный шаг учтен отдельно.


In [ ]:
fig, ax = plt.subplots(figsize=(10, 4), constrained_layout=True)
t_ref = np.linspace(t_switch, t_final, 1501)
ax.plot(t_ref, np.cos(t_ref), 'k-', lw=1.4, label='Точное u = cos(t)')
print(f"{'h, с':>8} {'Численная частота':>20} {'Отставание фазы, рад':>24} {'max|u - cos(t)|':>20}")
for h in [0.20, 0.50, 1.0, 2.0]:
    t2, y2 = continue_solution(step_trapezoidal, h)
    u2 = y2[:, 0] + y2[:, 2]
    omega_num = 2 / h * np.arctan(omega_slow_y * h / 2)
    phase_lag = (omega_slow_y * (t2[-1] - t2[0])
                 - np.sum(2 * np.arctan(omega_slow_y * np.diff(t2) / 2)))
    error = np.max(np.abs(u2 - np.cos(t2)))
    print(f'{h:8.2f} {omega_num:20.6f} {phase_lag:24.6f} {error:20.6f}')
    ax.plot(t2, u2, 'o--', ms=3, lw=1, label=f'Трапеции, h = {h} с')
ax.set(xlabel='Время t, с', ylabel='u = x1 + x2, м', title='Устойчивость не гарантирует правильную фазу')
ax.legend(fontsize=9)
plt.show()


---
## 4. Анимационные артефакты

### Анимация 1: Движение масс и пружин при слабой связи без демпфирования (биения)
На согласованной визуализации показаны:
- **Верхняя панель:** Физическая модель двух тележек на горизонтальной направляющей, соединенных пружинами с неподвижными стенками и между собой. Пружины сжимаются и растягиваются в реальном масштабе.
- **Нижняя панель:** Согласованные графики смещений $x_1(t)$ и $x_2(t)$ с бегущими маркерами, демонстрирующие переход амплитуды между массами.

Шаг между кадрами выбран не больше $T_{\min}/20\approx0.098$ с: на каждое колебание приходится не менее 20 кадров. При воспроизведении с 20 кадрами/с движение ускорено примерно вдвое. Полный период биений длинный, поэтому анимация строится дольше остальных графиков; прореживания кадров здесь нет.


In [ ]:
def draw_spring(x_start, x_end, y_pos=0.0, num_coils=10, width=0.12):
    """Генерация узлов ломаной линии для отрисовки деформируемой пружины."""
    length = x_end - x_start
    lead = 0.08 * length
    xs = [x_start, x_start + lead]
    ys = [y_pos, y_pos]
    coils_len = length - 2.0 * lead
    n_pts = 2 * num_coils
    for i in range(1, n_pts):
        xs.append(x_start + lead + coils_len * (i / n_pts))
        ys.append(y_pos + width * (1.0 if i % 2 == 1 else -1.0))
    xs.extend([x_end - lead, x_end])
    ys.extend([y_pos, y_pos])
    return np.array(xs), np.array(ys)

def build_beats_animation():
    """Построение анимации биений слабой связи."""
    # Моделируем один период биений
    # Не менее 20 кадров на самый короткий период, включая полное биение.
    shortest_period = 2 * np.pi / omega2_gr
    n_frames = int(np.ceil(T_beat_gr / (shortest_period / 20))) + 1
    t_anim = np.linspace(0.0, T_beat_gr, n_frames)
    x1_anim, _, x2_anim, _ = exact_solution_green(t_anim)
    
    # Равновесные координаты масс и опор
    L_wall_left = -3.5
    L_wall_right = 3.5
    eq1 = -1.2
    eq2 = 1.2
    mass_w = 0.6
    mass_h = 0.4
    
    frame_indices = np.arange(n_frames)
    
    fig, (ax_mech, ax_plot) = plt.subplots(2, 1, figsize=(10, 6), gridspec_kw={'height_ratios': [1, 1.2]})
    
    # 1. Настройка механической панели
    ax_mech.set_xlim(L_wall_left - 0.5, L_wall_right + 0.5)
    ax_mech.set_ylim(-0.8, 0.8)
    ax_mech.set_aspect('equal')
    ax_mech.axis('off')
    ax_mech.set_title("Механическая модель связанных масс и пружин (биения)", fontsize=11)
    
    # Стенки (опоры)
    ax_mech.plot([L_wall_left, L_wall_left], [-0.5, 0.5], 'k-', lw=4)
    ax_mech.plot([L_wall_right, L_wall_right], [-0.5, 0.5], 'k-', lw=4)
    
    # Пружины и массы
    spring1_line, = ax_mech.plot([], [], 'tab:blue', lw=1.5)
    spring_c_line, = ax_mech.plot([], [], 'tab:green', lw=1.5)
    spring2_line, = ax_mech.plot([], [], 'tab:blue', lw=1.5)
    
    cart1 = Rectangle((0, 0), mass_w, mass_h, fc='tab:blue', ec='black', lw=1.5, zorder=5)
    cart2 = Rectangle((0, 0), mass_w, mass_h, fc='tab:orange', ec='black', lw=1.5, zorder=5)
    ax_mech.add_patch(cart1)
    ax_mech.add_patch(cart2)
    
    # 2. Настройка графика смещений
    ax_plot.set_xlim(0, T_beat_gr)
    ax_plot.set_ylim(-1.3, 1.3)
    ax_plot.plot(t_anim, x1_anim, color='tab:blue', alpha=0.35, lw=1.0)
    ax_plot.plot(t_anim, x2_anim, color='tab:orange', alpha=0.35, lw=1.0)
    
    trace_x1, = ax_plot.plot([], [], color='tab:blue', lw=1.8, label=r'$x_1(t)$')
    trace_x2, = ax_plot.plot([], [], color='tab:orange', lw=1.8, label=r'$x_2(t)$')
    dot1, = ax_plot.plot([], [], 'o', color='tab:blue', ms=7)
    dot2, = ax_plot.plot([], [], 'o', color='tab:orange', ms=7)
    
    ax_plot.set_xlabel("Время $t$, с")
    ax_plot.set_ylabel("Смещение, м")
    ax_plot.legend(loc='upper right', fontsize=9)
    
    def update(frame):
        idx = frame_indices[frame]
        t_now = t_anim[idx]
        x1_now = x1_anim[idx]
        x2_now = x2_anim[idx]
        
        # Центры масс
        c1 = eq1 + x1_now
        c2 = eq2 + x2_now
        
        # Обновление масс
        cart1.set_xy((c1 - mass_w/2, -mass_h/2))
        cart2.set_xy((c2 - mass_w/2, -mass_h/2))
        
        # Обновление пружин
        s1_x, s1_y = draw_spring(L_wall_left, c1 - mass_w/2)
        sc_x, sc_y = draw_spring(c1 + mass_w/2, c2 - mass_w/2)
        s2_x, s2_y = draw_spring(c2 + mass_w/2, L_wall_right)
        
        spring1_line.set_data(s1_x, s1_y)
        spring_c_line.set_data(sc_x, sc_y)
        spring2_line.set_data(s2_x, s2_y)
        
        # Обновление графиков
        trace_x1.set_data(t_anim[:idx+1], x1_anim[:idx+1])
        trace_x2.set_data(t_anim[:idx+1], x2_anim[:idx+1])
        dot1.set_data([t_now], [x1_now])
        dot2.set_data([t_now], [x2_now])
        
        return spring1_line, spring_c_line, spring2_line, cart1, cart2, trace_x1, trace_x2, dot1, dot2

    anim = FuncAnimation(fig, update, frames=n_frames, interval=50, blit=True)
    plt.close(fig)
    return anim

anim_beats = build_beats_animation()
# Полное биение занимает около 3.4 минут просмотра; сохраняем все кадры.
with plt.rc_context({'animation.embed_limit': 256,
                     'animation.frame_format': 'jpeg', 'savefig.dpi': 60}):
    beats_html = anim_beats.to_jshtml()
HTML(beats_html)


### Анимация 2: Движение системы с сильной связью и демпфером
На согласованной визуализации показаны:
- **Верхняя панель:** Модель масс с жесткой связью и демпфером (символически изображенным поршнем между массами). Видно начальное быстрое относительное перемещение, которое за $\sim 1$ с затухает, после чего обе массы синхронно перемещаются вправо-влево как монолитный блок.
- **Нижняя панель:** Согласованный график относительного смещения $x_1(t) - x_2(t)$ с бегущей точкой во времени, подтверждающий быстрое экспоненциальное затухание относительной моды.

На участке $0\le t\le1$ с шаг между кадрами равен $0.008$ с: это около 49 кадров на период $2\pi/\Omega$ и 10 кадров на время затухания $1/\gamma$. Этот участок воспроизводится с пятикратным замедлением, затем идет в реальном времени. Текущее модельное время и скорость воспроизведения подписаны над системой.


In [ ]:
def build_damped_animation():
    """Построение анимации системы с сильной связью и демпфером."""
    # Первую секунду показываем в 5 раз медленнее реального времени.
    # По 10 кадров на время затухания и не менее 20 на быстрый период.
    fps = 25
    t_early = np.linspace(0.0, 1.0, 126)
    t_late = np.linspace(1.0, 8.0, 176)[1:]
    t_anim = np.concatenate((t_early, t_late))
    n_frames = len(t_anim)
    # Точное решение исключает дополнительную ошибку интерполяции.
    y_anim = exact_solution_y(t_anim)
    x1_anim, x2_anim = y_anim[:, 0], y_anim[:, 2]
    rel_x = x1_anim - x2_anim

    L_wall_left = -3.5
    L_wall_right = 3.5
    eq1 = -1.2
    eq2 = 1.2
    mass_w = 0.6
    mass_h = 0.4
    
    frame_indices = np.arange(n_frames)
    fig, (ax_mech, ax_plot) = plt.subplots(2, 1, figsize=(10, 6), gridspec_kw={'height_ratios': [1, 1.2]})
    
    ax_mech.set_xlim(L_wall_left - 0.5, L_wall_right + 0.5)
    ax_mech.set_ylim(-0.8, 0.8)
    ax_mech.set_aspect('equal')
    ax_mech.axis('off')
    ax_mech.set_title("Система с сильной связью и демпфером (быстрое затухание относительной моды)", fontsize=11)
    
    ax_mech.plot([L_wall_left, L_wall_left], [-0.5, 0.5], 'k-', lw=4)
    ax_mech.plot([L_wall_right, L_wall_right], [-0.5, 0.5], 'k-', lw=4)
    
    spring1_line, = ax_mech.plot([], [], 'tab:blue', lw=1.5)
    spring_c_line, = ax_mech.plot([], [], 'tab:red', lw=2.0)
    spring2_line, = ax_mech.plot([], [], 'tab:blue', lw=1.5)
    damper_line, = ax_mech.plot([], [], 'k-', lw=2.5)  # демпфер
    
    cart1 = Rectangle((0, 0), mass_w, mass_h, fc='tab:blue', ec='black', lw=1.5, zorder=5)
    cart2 = Rectangle((0, 0), mass_w, mass_h, fc='tab:blue', ec='black', lw=1.5, zorder=5)
    ax_mech.add_patch(cart1)
    ax_mech.add_patch(cart2)
    
    time_text = ax_mech.text(0.02, 0.93, '', transform=ax_mech.transAxes, fontsize=9)
    ax_plot.set_xlim(0, 8.0)
    ax_plot.set_ylim(-0.25, 1.1)
    ax_plot.plot(t_anim, rel_x, color='tab:purple', alpha=0.35, lw=1.0)
    trace_rel, = ax_plot.plot([], [], color='tab:purple', lw=1.8, label=r'Относительное смещение $x_1(t) - x_2(t)$')
    dot_rel, = ax_plot.plot([], [], 'o', color='tab:red', ms=7)
    
    ax_plot.axhline(0, color='gray', ls=':', alpha=0.6)
    ax_plot.set_xlabel("Время $t$, с")
    ax_plot.set_ylabel(r"$x_1 - x_2$, м")
    ax_plot.legend(loc='upper right', fontsize=9)
    
    def update(frame):
        idx = frame_indices[frame]
        t_now = t_anim[idx]
        speed_note = 'замедление в 5 раз' if t_now <= 1.0 else 'реальное время'
        time_text.set_text(f't = {t_now:.3f} с; {speed_note}')
        x1_now = x1_anim[idx]
        x2_now = x2_anim[idx]
        
        c1 = eq1 + x1_now
        c2 = eq2 + x2_now
        
        cart1.set_xy((c1 - mass_w/2, -mass_h/2))
        cart2.set_xy((c2 - mass_w/2, -mass_h/2))
        
        s1_x, s1_y = draw_spring(L_wall_left, c1 - mass_w/2, y_pos=0.0)
        sc_x, sc_y = draw_spring(c1 + mass_w/2, c2 - mass_w/2, y_pos=0.1)
        s2_x, s2_y = draw_spring(c2 + mass_w/2, L_wall_right, y_pos=0.0)
        
        # Демпфер (поршень параллельно пружине связи)
        d_x = [c1 + mass_w/2, c1 + mass_w/2 + 0.4*(c2 - c1 - mass_w),
               c1 + mass_w/2 + 0.4*(c2 - c1 - mass_w), c2 - mass_w/2]
        d_y = [-0.1, -0.1, -0.1, -0.1]
        
        spring1_line.set_data(s1_x, s1_y)
        spring_c_line.set_data(sc_x, sc_y)
        spring2_line.set_data(s2_x, s2_y)
        damper_line.set_data(d_x, d_y)
        
        trace_rel.set_data(t_anim[:idx+1], rel_x[:idx+1])
        dot_rel.set_data([t_now], [rel_x[idx]])
        
        return spring1_line, spring_c_line, spring2_line, damper_line, cart1, cart2, trace_rel, dot_rel, time_text

    anim = FuncAnimation(fig, update, frames=n_frames, interval=1000 / fps, blit=True)
    plt.close(fig)
    return anim

anim_damped = build_damped_animation()
with plt.rc_context({'animation.embed_limit': 128, 'savefig.dpi': 80}):
    damped_html = anim_damped.to_jshtml()
HTML(damped_html)


---
## 5. Ответ на вопрос и выводы

### Ответ на вопрос: *«Можно ли после затухания быстрого процесса увеличить шаг, сохранив устойчивость и точность медленного движения?»*

Шаг увеличить можно в обоих методах, но ограничения различаются.

**RK4.** Для полной системы требуется $|R_{\mathrm{RK4}}(h\lambda_j)|\le1$ для всех собственных значений. Расчет по спектру дает $h_*\approx0.131258$ с. Поэтому переход с $h=0.005$ на $0.12$ или $0.13$ с допустим по устойчивости. При $h=0.20$ с коэффициент усиления быстрой моды равен примерно $5.70$: даже малый остаток этой моды начинает расти. В точной арифметике подпространство $x_1=x_2$, $v_1=v_2$ инвариантно и содержит только медленную моду; однако после конечного времени затухания решение еще не лежит в нем точно. В вычислениях также действуют ошибки округления. Поэтому для расчета исходной системы нельзя опираться только на частоту медленного движения.

**Метод трапеций.** Устойчивость сохраняется при любом положительном шаге, но частота численного решения отличается от точной. Для медленной моды
$$R(i\omega h)=\exp\bigl(2i\arctan(\omega h/2)\bigr),\qquad
\widetilde\omega=\frac{2}{h}\arctan\frac{\omega h}{2}<\omega.$$
Амплитуда сохраняется, а фаза постепенно отстает. При $\omega=1$, $h=0.20$ с за $13.5$ с после переключения отставание составляет около $0.044$ рад; при $h=1$ с — уже $0.955$ рад (с учетом последнего укороченного шага). В последнем случае решение ограничено, но заметно неверно передает колебания. Допустимый шаг следует выбирать по требуемой фазовой точности на всем интервале наблюдения, например проверять $|\omega-\widetilde\omega|(t_{\mathrm{final}}-t_{\mathrm{switch}})$ и сравнивать расчет с вдвое меньшим шагом.

Для быстрой моды тоже нужно проверять фазу и затухание: метод трапеций A-устойчив, но не L-устойчив ($R(z)\to-1$ при $|z|\to\infty$ в левой полуплоскости). На крупном шаге он может медленно гасить численные колебания вместо быстрого физического затухания. В этом примере сначала нужен мелкий шаг для переходного процесса; после его затухания шаг можно увеличить с контролем ошибки медленной моды.

---

### Общие выводы по работе

1. **Биения при слабой связи:**
   - Успешно промоделировано движение связанных осцилляторов методами RK4 и Верле при $k_c / k_0 = 0.01$.
   - Обнаружено четкое расщепление спектра на две собственные частоты $\omega_1 = 3.1623$ рад/с и $\omega_2 = 3.1937$ рад/с, что дает аналитический период биений $T_{\mathrm{б}} = 399.36$ с. Численный спектр Фурье (FFT) подтвердил теоретические значения с точностью до $10^{-4}$ рад/с.
   - Симплектический скоростной метод Верле продемонстрировал идеальную долговременную стабильность квазиинтеграла энергии без численного дрейфа, в то время как RK4 обладает монотонным дрейфом погрешности $O(h^4)$.
   - Создана согласованная анимация физического движения масс на пружинах и графика биений.

2. **Жёсткость и неявные методы:**
   - Подобраны параметры сильной связи и вязкого демпфирования ($k_c = 200$, $c = 12$), обеспечивающие быстрое колебательное затухание относительного смещения за время $\tau \sim 0.08$ с при сохранении чистых совместных синфазных колебаний.
   - Для RK4 проверена граница устойчивости $h_*\approx0.131258$ с: после затухания быстрого процесса шаг можно увеличить до $0.12$–$0.13$ с, но при $0.20$ с остаточная быстрая мода растет.
   - Для метода трапеций AM2 увеличение шага ограничено точностью фазы и воспроизведения затухания, несмотря на безусловную устойчивость. Проверено накопление фазовой ошибки при крупных шагах.
   - Разработана интерактивная анимация демпфированной системы с синхронизированным графиком относительного смещения $x_1 - x_2$.
